In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

# CONFIG (edit these 3 lines only)
PATH = "data/entsog_sample.csv"
COL_T, COL_Y, COL_YHAT = "timestamp", "y", "y_hat"

df = pd.read_csv(PATH)
df[COL_T] = pd.to_datetime(df[COL_T])
df = df.sort_values(COL_T).dropna(subset=[COL_Y, COL_YHAT]).reset_index(drop=True)

df.head(), df.shape

In [ ]:
# rolling residual volatility model (very baseline)
resid = df[COL_Y] - df[COL_YHAT]

# use a rolling window (e.g., 7 days of quarter-hourly -> 7*96 = 672)
W = 7 * 96
sigma = resid.rolling(W, min_periods=W).std()

# align: only evaluate when sigma exists
mask = sigma.notna()
y = df.loc[mask, COL_Y].to_numpy()
yhat = df.loc[mask, COL_YHAT].to_numpy()
sig = sigma.loc[mask].to_numpy()

# PIT: u_t = Phi((y - yhat)/sig)
z = (y - yhat) / sig
u = norm.cdf(z)

u[:5], u.shape

In [ ]:
# PIT histogram
plt.figure()
plt.hist(u, bins=20)
plt.title("PIT histogram (baseline residual Normal)")
plt.show()

# PIT autocorrelation (lag 1..50)
def autocorr(x, lag):
    x = x - x.mean()
    return np.corrcoef(x[:-lag], x[lag:])[0,1]

acs = [autocorr(u, k) for k in range(1, 51)]
plt.figure()
plt.plot(range(1, 51), acs)
plt.title("PIT autocorrelation")
plt.xlabel("Lag")
plt.show()

# coverage check for central intervals (80%, 90%)
for alpha in [0.2, 0.1]:
    lo = norm.ppf(alpha/2, loc=yhat, scale=sig)
    hi = norm.ppf(1-alpha/2, loc=yhat, scale=sig)
    cov = np.mean((y >= lo) & (y <= hi))
    print(f"Nominal {int((1-alpha)*100)}% coverage: empirical {cov:.3f}")